# Week 4 · PCA in pictures

The first unsupervised method: there are no labels, only $X$. **Principal component analysis (PCA)** finds the directions along which the data spreads the most, and describes every point by its position along a few of them. It's how you go from 30 or 64 features down to 2 for a picture, or to 10 for a faster, less noisy model. As in Week 3's logistic regression notebook, one small dataset (5 points) is carried through every section, with every formula drawn and every number computed. **Reading:** ISL Ch. 12.1–12.2 (it's the PDF in your repo); StatQuest "PCA main ideas" and "PCA step by step".

**How to use this notebook**
- Run cells one at a time with **Shift + Enter**.
- Before running an example, **guess what it prints**. The real result is written after `# →` in the comments, so you can check yourself.
- **Your turn** cells are for you. Open **Show one answer** only after a real attempt.
- If things get weird, click **Restart** at the top and run from the top again.

## Words you'll see
- **principal component (PC)**: A direction in feature space (a unit vector). PC1 is the direction of most spread, PC2 the most spread at right angles to PC1, and so on.
- **score**: A point's coordinate along a PC: its shadow length $u^\top (x - \bar x)$.
- **explained variance**: How much of the data's total spread one PC accounts for. It's that PC's eigenvalue $\lambda$.
- **unsupervised**: Learning from $X$ alone, with no $y$. PCA and clustering are the two main kinds.

**Run this cell first.** It loads NumPy, the picture helpers (`figures.py` in this folder) and the exercise checker.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from figures import *
import checks; checks.load(globals(), "02")      # defines check(n)

## M. The map: what's in this notebook and why

🖼 **Picture.** PCA as a pipeline. The top row finds the directions; the bottom row uses them.

In [ ]:
fig_pca_map()

| Section | What | Why it's here |
|---|---|---|
| 1 | the question | of all directions, which one keeps the points most spread out? |
| 2 | center | PCA measures spread around the mean, so the mean goes to (0, 0) first |
| 3 | spread along any direction | a formula, $u^\top S u$, that scores every direction at once |
| 4 | the covariance matrix $S$ | the table of numbers that formula needs |
| 5 | the best direction is an eigenvector | turns "try every direction" into one calculation; the exam's 2×2 hand drill |
| 6 | project: the new coordinates | each point described by its scores along the PCs |
| 7 | how many components? | explained variance and the 90% rule |
| 8 | reconstruct | what's lost when you keep fewer components; how PCA differs from regression |
| 9 | the same thing via SVD | what scikit-learn actually computes |
| 10–11 | scaling, real data | the two things that go wrong in practice, and PCA on 30 and 64 features |
| 12 | beyond PCA | kernel PCA, t-SNE, LDA: when a straight-line projection isn't enough |
| 13 | build it yourself | 8 short exercises |

**How it connects:** PCA uses three things you already have: projection lengths ([Week 2 · 01 §4 ↗](../week2/01_linear_algebra_review.ipynb)), variance ([Week 1 · math §2 ↗](../week1/03_math_essentials.ipynb)) and eigenvectors ([Week 1 · math §6 ↗](../week1/03_math_essentials.ipynb)). Clustering (`03`, `04`) is the other half of unsupervised learning: PCA simplifies the columns, clustering groups the rows.

## 0. Recap first · 10 min, closed book

**Quick check:** The length of the projection (shadow) of $b$ onto a **unit** vector $u$?

<details><summary>Answer</summary>

$u^\top b$. For a non-unit $a$ it's $\frac{a^\top b}{\lVert a \rVert}$. [Week 2 · 01 §4 ↗](../week2/01_linear_algebra_review.ipynb)

</details>

**Quick check:** Sample variance of $1, 2, 3, 4, 5$ (divide by $n - 1$)?

<details><summary>Answer</summary>

Mean 3; squared deviations $4, 1, 0, 1, 4$, sum 10; $\frac{10}{4} = 2.5$. [Week 1 · math §2 ↗](../week1/03_math_essentials.ipynb)

</details>

**Quick check:** What does $Av = \lambda v$ say in words?

<details><summary>Answer</summary>

Multiplying $v$ by $A$ only stretches it (by $\lambda$); it doesn't turn it. $v$ is an eigenvector, $\lambda$ its eigenvalue. [Week 1 · math §6 ↗](../week1/03_math_essentials.ipynb)

</details>

**Quick check:** `X` has shape `(100, 5)`. Shape of `X.mean(axis=0)`, and of `X - X.mean(axis=0)`?

<details><summary>Answer</summary>

`(5,)`, one mean per feature; `(100, 5)`, broadcasting subtracts each column's mean. [Week 3 · 02 §8 ↗](../week3/02_knn_and_numpy_workout.ipynb)

</details>

**Quick check:** $\hat p = \sigma(z)$ with threshold 0.8. At which height $z$ is the decision line?

<details><summary>Answer</summary>

$\ln\frac{0.8}{0.2} = 1.386$. [Week 3 · 01 §9 ↗](../week3/01_logistic_regression_in_pictures.ipynb)

</details>

## 1. The question: which direction keeps the points most spread out?

**PCA looks for the direction along which the data varies the most. Describing each point by its position along that direction keeps as much of the information as one number can.**

**Our data:** 5 points with 2 features each. They clearly lie roughly along a diagonal.

|  | P1 | P2 | P3 | P4 | P5 |
|---|---|---|---|---|---|
| $x_1$ | 3 | 4 | 5 | 6 | 7 |
| $x_2$ | 3 | 2 | 4 | 6 | 5 |

In [ ]:
XP = np.array([[3, 3], [4, 2], [5, 4], [6, 6], [7, 5]], dtype=float)    # (5, 2): 5 points, 2 features

🖼 **Picture.** The (centred) points projected onto three lines through the middle. Along 45° the shadows are spread far apart (variance 4.5); along 135° they're bunched up (0.5).

In [ ]:
fig_pca_directions()

If you had to describe each point with **one number**, its shadow on the 45° line would be the best choice: the shadows differ a lot, so they still tell the points apart. On the 135° line, the shadows are almost all the same and say little. **PCA finds that best line, and then the next best at right angles to it.**

## 2. Step 1: center the data

**Subtract the mean of each feature, so the cloud of points sits around the origin.** Spread is measured from the mean, and the projection formula in §3 assumes the lines pass through it.

$$
\bar x = \frac{1}{n}\sum_{i=1}^n x_i, \qquad x_i^{c} = x_i - \bar x
$$

Here $\bar x = \big(\frac{3 + 4 + 5 + 6 + 7}{5}, \frac{3 + 2 + 4 + 6 + 5}{5}\big) = (5, 4)$, so the centred points are $(-2, -1), (-1, -2), (0, 0), (1, 2), (2, 1)$.

🖼 **Picture.** The same points before and after centring. Nothing about their shape changes; they just move so the mean is at (0, 0).

In [ ]:
fig_center()

In [ ]:
mean = XP.mean(axis=0)            # (2,)   one mean per feature
Xc = XP - mean                    # (5, 2) broadcasting: each row minus the mean
print(mean, Xc.tolist())
# → [5. 4.] [[-2.0, -1.0], [-1.0, -2.0], [0.0, 0.0], [1.0, 2.0], [2.0, 1.0]]

## 3. Step 2: the spread along any direction

**The variance of the shadows along a unit vector $u$ is $u^\top S u$, where $S$ is the covariance matrix.** One formula scores every possible direction.

**Where it comes from.** Each centred point's shadow on $u$ is $s_i = u^\top x_i^c$, so all the shadows at once are $s = X_c u$, shape `(n,)`. They have mean 0 (the data is centred), so their variance is

$$
\operatorname{Var}(s) = \frac{1}{n - 1}\sum_i s_i^2 = \frac{1}{n - 1}\, s^\top s = \frac{1}{n - 1}\, u^\top X_c^\top X_c\, u = u^\top \underbrace{\Big(\frac{1}{n - 1} X_c^\top X_c\Big)}_{S}\, u
$$

| Direction | $u$ | Shadows $X_c u$ | Variance |
|---|---|---|---|
| 0° | (1, 0) | −2, −1, 0, 1, 2 | $\frac{4 + 1 + 0 + 1 + 4}{4} = 2.5$ |
| 45° | (0.707, 0.707) | −2.12, −2.12, 0, 2.12, 2.12 | $\frac{4 \times 4.5}{4} = 4.5$ |
| 90° | (0, 1) | −1, −2, 0, 2, 1 | 2.5 |
| 135° | (−0.707, 0.707) | 0.71, −0.71, 0, 0.71, −0.71 | $\frac{4 \times 0.5}{4} = 0.5$ |

The 45° row by hand: P1's shadow is $0.707 \times (-2) + 0.707 \times (-1) = -2.121$, and $2.121^2 = 4.5$. Four points have shadows of size 2.121 and one has 0, so the variance is $\frac{4 \times 4.5}{4} = 4.5$.

🖼 **Picture.** $u^\top S u$ for every direction from 0° to 180°. One clear peak (45°) and one clear valley (135°), exactly 90° apart.

In [ ]:
fig_variance_by_angle()

## 4. The covariance matrix

**The covariance matrix $S$ holds each feature's variance on the diagonal and the covariance of each pair of features off the diagonal.**

$$
S = \frac{1}{n - 1} X_c^\top X_c, \qquad S_{jk} = \frac{1}{n - 1}\sum_i x^c_{ij}\, x^c_{ik}
$$

$S$ is $d \times d$ (here $2 \times 2$) and symmetric, $S_{jk} = S_{kj}$. A positive off-diagonal entry means the two features tend to go up together.

| Point | $x^c_1$ | $x^c_2$ | $(x^c_1)^2$ | $(x^c_2)^2$ | $x^c_1\, x^c_2$ |
|---|---|---|---|---|---|
| P1 | −2 | −1 | 4 | 1 | 2 |
| P2 | −1 | −2 | 1 | 4 | 2 |
| P3 | 0 | 0 | 0 | 0 | 0 |
| P4 | 1 | 2 | 1 | 4 | 2 |
| P5 | 2 | 1 | 4 | 1 | 2 |
| **sum** |  |  | **10** | **10** | **8** |

$$
S = \frac{1}{4}\begin{pmatrix} 10 & 8 \\ 8 & 10 \end{pmatrix} = \begin{pmatrix} 2.5 & 2 \\ 2 & 2.5 \end{pmatrix}
$$

In [ ]:
S = Xc.T @ Xc / (len(Xc) - 1)     # (2, 5) @ (5, 2) → (2, 2)
print(S)
# → [[2.5 2. ]
#    [2.  2.5]]
print(np.allclose(S, np.cov(XP.T)))   # → True   np.cov wants one ROW per feature, hence .T

Check against §3: $u = (1, 0)$ gives $u^\top S u = S_{11} = 2.5$ ✓, and $u = (0.707, 0.707)$ gives $0.5 \times (2.5 + 2 + 2 + 2.5) = 4.5$ ✓.

## 5. Step 3: the best direction is the top eigenvector

**The direction of maximum variance is the eigenvector of $S$ with the largest eigenvalue, and that eigenvalue is the variance along it.** The next components are the next eigenvectors, which are automatically at right angles.

**Where it comes from.** We want the unit vector that maximizes $u^\top S u$, subject to $u^\top u = 1$ (without that constraint, longer $u$ would always win). With a Lagrange multiplier $\lambda$:

$$
\mathcal{L}(u, \lambda) = u^\top S u - \lambda\,(u^\top u - 1), \qquad \nabla_u \mathcal{L} = 2Su - 2\lambda u = 0 \;\Longrightarrow\; S u = \lambda u
$$

So the best $u$ must be an **eigenvector** of $S$. Which one? The variance along it is $u^\top S u = u^\top (\lambda u) = \lambda\, u^\top u = \lambda$: **the eigenvalue is the variance**. Pick the eigenvector with the largest $\lambda$.

**The exam drill: eigen-decompose a 2×2 covariance by hand**

- Eigenvalues solve $\det(S - \lambda I) = 0$: $(2.5 - \lambda)^2 - 2 \cdot 2 = 0$, so $2.5 - \lambda = \pm 2$, so $\lambda_1 = 4.5$ and $\lambda_2 = 0.5$.
- Eigenvector for $\lambda_1 = 4.5$: $(S - 4.5 I)\,v = \begin{pmatrix} -2 & 2 \\ 2 & -2 \end{pmatrix} v = 0$ means $v_1 = v_2$, so $v \propto (1, 1)$. Unit length: $u_1 = (0.707, 0.707)$.
- Eigenvector for $\lambda_2 = 0.5$: $\begin{pmatrix} 2 & 2 \\ 2 & 2 \end{pmatrix} v = 0$ means $v_1 = -v_2$, so $u_2 = (-0.707, 0.707)$. It's at right angles to $u_1$: $u_1^\top u_2 = 0$.
- Check the total: $\lambda_1 + \lambda_2 = 5 = 2.5 + 2.5$, the sum of the diagonal (the trace). PCA moves variance between directions, but never creates or destroys it.

In [ ]:
values, vectors = np.linalg.eigh(S)     # eigh: for symmetric matrices; returns the SMALLEST eigenvalue first
print(values)                            # → [0.5 4.5]
order = np.argsort(values)[::-1]         # largest first
values, W = values[order], vectors[:, order]
print(values, W[:, 0].round(3))          # → [4.5 0.5] [0.707 0.707]   column 0 = PC1

> ⚠️ **Common mistake.** **The sign of an eigenvector is arbitrary.** $(0.707, 0.707)$ and $(-0.707, -0.707)$ are the same direction. Two programs (or two runs) can return opposite signs, and then all the scores flip sign too. Nothing is wrong; compare directions up to sign.

## 6. Step 4: project, i.e. the new coordinates

**A point's scores are its shadows on the principal components. Stacking the top $k$ components as the columns of $W$ (shape $d \times k$) gives every score at once: $Z = X_c W$, shape $n \times k$.**

| Point | Centred | Score on PC1, $u_1^\top x^c$ | Score on PC2, $u_2^\top x^c$ |
|---|---|---|---|
| P1 | (−2, −1) | $0.707(-2 - 1) = -2.121$ | $0.707(2 - 1) = 0.707$ |
| P2 | (−1, −2) | −2.121 | −0.707 |
| P3 | (0, 0) | 0 | 0 |
| P4 | (1, 2) | 2.121 | 0.707 |
| P5 | (2, 1) | 2.121 | −0.707 |

🖼 **Picture.** The new axes drawn on the centred data: PC1 (orange) along the spread, PC2 (green) across it. The arrows are scaled by $\sqrt{\lambda}$ so their length shows the spread. Grey numbers are each point's scores.

In [ ]:
fig_pca_axes()

Rotating to these axes changes nothing about the points; it's the same picture tilted by 45°. The point is that in the new axes **almost all the variation is in the first coordinate** (4.5 out of 5), so dropping the second loses little.

In [ ]:
Z = Xc @ W                    # (5, 2) @ (2, 2) → (5, 2): every point's scores
print(Z.round(3))
# → [[-2.121  0.707]
#    [-2.121 -0.707]
#    [ 0.     0.   ]
#    [ 2.121  0.707]
#    [ 2.121 -0.707]]
print(Z.var(axis=0, ddof=1).round(3))   # → [4.5 0.5]   the score variances ARE the eigenvalues

## 7. How many components to keep

**The share of the total variance kept by the first $k$ components is $\frac{\lambda_1 + \dots + \lambda_k}{\lambda_1 + \dots + \lambda_d}$.** A common rule: keep enough components for 90%.

Here PC1 alone keeps $\frac{4.5}{4.5 + 0.5} = 90\%$, so one number per point is enough. On real data the eigenvalues fall off gradually, and a **scree plot** shows where:

🖼 **Picture.** The breast cancer data (30 standardized features). PC1 keeps 44%, PC1 + PC2 keep 63%, and 7 components reach 90%.

In [ ]:
fig_scree()

Other ways to choose $k$: look for the "elbow" where the bars flatten, or, if PCA feeds a model, choose $k$ by cross-validation like any other setting (§11).

## 8. Reconstructing, and how PCA differs from regression

**From $k$ scores you can rebuild an approximation of each point: start at the mean and walk each score's distance along its component, $\hat x = \bar x + \sum_{j=1}^{k} z_j u_j$.** In matrix form, $\hat X = Z W^\top + \bar x$.

With only PC1: P1 has score $-2.121$, so $\hat x = (5, 4) + (-2.121)(0.707, 0.707) = (5, 4) - (1.5, 1.5) = (3.5, 2.5)$. The original was $(3, 3)$.

🖼 **Picture.** Each point squashed onto the PC1 line (green squares). The red gaps are what one number per point can't capture.

In [ ]:
fig_reconstruction()

**The lost part is exactly the dropped variance.** Each gap is the point's PC2 score: four points are $0.707$ off the line, and $4 \times 0.707^2 = 4 \times 0.5 = 2.0$, which equals $\lambda_2 \times (n - 1) = 0.5 \times 4$. In general, PCA with $k$ components gives the **smallest possible** total squared reconstruction error of any $k$-dimensional projection.

🖼 **Picture.** Same cloud, two different lines. Regression minimizes **vertical** gaps, because it predicts $x_2$ from $x_1$. PCA minimizes **perpendicular** gaps, because it treats every feature the same and predicts nothing.

In [ ]:
fig_regression_vs_pca()

In [ ]:
W1 = W[:, :1]                         # (2, 1): keep PC1 only
Z1 = Xc @ W1                          # (5, 1)
X_hat = Z1 @ W1.T + mean              # (5, 1) @ (1, 2) + (2,) → (5, 2)
print(X_hat.round(2).tolist())        # → [[3.5, 2.5], [3.5, 2.5], [5.0, 4.0], [6.5, 5.5], [6.5, 5.5]]
print(round(((XP - X_hat) ** 2).sum(), 3))   # → 2.0

## 9. The same thing via SVD

**The singular value decomposition $X_c = U \Sigma V^\top$ gives the principal components directly: they are the rows of $V^\top$, and the eigenvalues are $\lambda_j = \frac{\sigma_j^2}{n - 1}$.**

**Why it works.** Substitute the SVD into the covariance: $S = \frac{1}{n - 1} X_c^\top X_c = \frac{1}{n - 1} V \Sigma U^\top U \Sigma V^\top = V \frac{\Sigma^2}{n - 1} V^\top$, because $U^\top U = I$. That's an eigen-decomposition of $S$, with eigenvectors $V$ and eigenvalues $\frac{\sigma_j^2}{n - 1}$. scikit-learn's `PCA` uses the SVD because it never forms $X_c^\top X_c$, which is more accurate and faster when $d$ is large.

In [ ]:
U, sigma, Vt = np.linalg.svd(Xc, full_matrices=False)
print(sigma.round(4), (sigma ** 2 / 4).round(3))   # → [4.2426 1.4142] [4.5 0.5]
from sklearn.decomposition import PCA
pca = PCA(n_components=2).fit(XP)                  # centres the data for you
print(pca.components_.round(3))                   # rows = PCs (sign may differ)
# → [[ 0.707  0.707]
#    [-0.707  0.707]]
print(pca.explained_variance_, pca.explained_variance_ratio_)   # → [4.5 0.5] [0.9 0.1]

Note the shapes: `pca.components_` has one PC per **row**, shape $k \times d$, which is $W^\top$. And `pca.transform(X)` returns the scores $Z$.

## 10. Scale the features first (usually)

**PCA chases variance, so a feature measured in big units dominates just because its numbers are big.** When the features have different units, standardize them first (mean 0, std 1 for each).

🖼 **Picture.** Age (years) and income (thousands of won). Unscaled, PC1 is almost exactly the income axis: income's variance is a million times bigger. Standardized, PC1 runs along the real diagonal relationship.

In [ ]:
fig_scaling_pca()

In the breast cancer data, without scaling, PC1 "explains" 98% of the variance, but only because `worst area` has a standard deviation of 569 while some features vary by 0.003. Standardizing gives the honest 44% from §7. **Exception:** when every feature is in the same unit and scale matters (pixels, for example), don't scale.

## 11. PCA on real data

🖼 **Picture.** Left: 569 tumours, 30 features each, squashed to 2 scores. PCA never saw the labels, yet malignant and benign separate along PC1. Right: 1797 images of handwritten digits, 64 pixels each, squashed to 2 scores.

In [ ]:
fig_pca_real()

**PCA before a model** keeps the useful directions and drops the noise in the rest. Put it in a pipeline so it's fitted on the training folds only, and choose $k$ by cross-validation:

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
Xb, yb = load_breast_cancer(return_X_y=True)
Xa, Xt, ya, yt = train_test_split(Xb, yb, test_size=0.25, random_state=0, stratify=yb)
for k in [1, 2, 5, 10, 30]:
    model = make_pipeline(StandardScaler(), PCA(k), LogisticRegression(max_iter=1000))
    print(k, round(cross_val_score(model, Xa, ya, cv=5).mean(), 3))
# → 1 0.916
#   2 0.963
#   5 0.974
#   10 0.981
#   30 0.974

Two components already reach 96%, and 10 components (a third of the features) score slightly **better** than all 30: dropping the low-variance directions removed some noise. That's the variance side of the bias–variance trade-off again.

## 12. Beyond PCA (the ideas, briefly)

PCA can only rotate and squash along **straight** directions. Three relatives cover what it can't:

| Method | Idea | Use it for |
|---|---|---|
| kernel PCA | PCA after an implicit feature map, via a kernel, exactly as in the kernel SVM (00 §B2) | curved structure (circles, spirals) |
| t-SNE, UMAP | place points in 2-D so that **neighbours stay neighbours**; distances between far-apart groups mean little | **pictures only**: exploring clusters, never as model input |
| LDA (Fisher's linear discriminant) | **supervised**: the direction that best **separates the classes**, not the one with most spread | a 1–2-D projection when you have labels |

🖼 **Picture.** The same 900 digit images in 2-D. PCA (left) is a straight projection: the digits overlap. t-SNE (right) pulls each digit's images into its own island.

In [ ]:
fig_pca_vs_tsne()

🖼 **Picture.** Two classes that are long and thin and sit side by side. PCA picks the long direction (most spread), where the classes overlap completely. LDA uses the labels and picks the direction across them, where they separate.

In [ ]:
fig_pca_vs_lda()

## 13. Build it yourself

Conventions: `X` is `(n, d)`; `W` holds one component per **column**, `(d, k)`; scores `Z` are `(n, k)`.

✏️ **Exercise 1 · Center.** Subtract each feature's mean. **Shape `(n, d)`.**

**Example**: what goes in, and what should come out:
```python
XP = np.array([[3, 3], [4, 2], [5, 4], [6, 6], [7, 5]], dtype=float)
center(XP)     # → [[-2. -1.] [-1. -2.] [ 0.  0.] [ 1.  2.] [ 2.  1.]]
```

In [ ]:
def center(X):
    pass  # your code

In [ ]:
check(1)

✏️ **Exercise 2 · Covariance.** $S = \frac{1}{n - 1} X_c^\top X_c$ for already-centred `Xc`. **Shape `(d, d)`.** (Don't use `np.cov`.)

**Example**: what goes in, and what should come out:
```python
covariance(center(XP))     # → [[2.5 2. ]
                            #    [2.  2.5]]
```

In [ ]:
def covariance(Xc):
    pass  # your code

In [ ]:
check(2)

✏️ **Exercise 3 · Eigen-decomposition, largest first.** Return `(values, vectors)` from `np.linalg.eigh(S)`, reordered so the **largest** eigenvalue comes first; reorder the **columns** of `vectors` the same way. Shapes `(d,)` and `(d, d)`.

**Example**: what goes in, and what should come out:
```python
values, vectors = pca_eig(np.array([[2.5, 2.0], [2.0, 2.5]]))
values     # → [4.5 0.5]
vectors    # → [[ 0.707 -0.707]
           #    [ 0.707  0.707]]     column 0 = PC1 (each column's sign may be flipped)
```

In [ ]:
def pca_eig(S):
    pass  # your code

In [ ]:
check(3)

✏️ **Exercise 4 · Project.** Scores $Z = X_c W$. **Shape `(n, k)`.**

**Example**: what goes in, and what should come out:
```python
W1 = np.array([[0.7071], [0.7071]])     # (2, 1): PC1 as a column
project(center(XP), W1)                  # → [[-2.121] [-2.121] [ 0.   ] [ 2.121] [ 2.121]]   shape (5, 1)
```

In [ ]:
def project(Xc, W):
    pass  # your code

In [ ]:
check(4)

✏️ **Exercise 5 · Reconstruct.** $\hat X = Z W^\top + \bar x$. **Shape `(n, d)`.** With all $d$ components it gives back `X` exactly.

**Example**: what goes in, and what should come out:
```python
Z1 = project(center(XP), W1)
reconstruct(Z1, W1, np.array([5.0, 4.0]))
# → [[3.5 2.5] [3.5 2.5] [5.  4. ] [6.5 5.5] [6.5 5.5]]
```

In [ ]:
def reconstruct(Z, W, mean):
    pass  # your code

In [ ]:
check(5)

✏️ **Exercise 6 · Explained variance ratio.** Each eigenvalue divided by their sum. **Shape `(d,)`.**

**Example**: what goes in, and what should come out:
```python
explained_ratio(np.array([4.5, 0.5]))     # → [0.9 0.1]
```

In [ ]:
def explained_ratio(values):
    pass  # your code

In [ ]:
check(6)

✏️ **Exercise 7 · How many components.** `values` are eigenvalues, sorted largest first. Return the smallest $k$ whose cumulative share of the total reaches `target` (e.g. 0.9). Hint: `np.cumsum`, then the first position where it's `>= target`.

**Example**: what goes in, and what should come out:
```python
n_components_for(np.array([4.5, 0.5]), 0.9)        # → 1   (4.5 / 5 = 0.9 is enough)
n_components_for(np.array([5.0, 3.0, 2.0]), 0.8)   # → 2   cumulative shares 0.5, 0.8, 1.0
```

In [ ]:
def n_components_for(values, target):
    pass  # your code

In [ ]:
check(7)

✏️ **Exercise 8 · PCA via SVD.** Center `X`, take `np.linalg.svd(Xc, full_matrices=False)`, keep the first `k` rows of $V^\top$ as `components` (shape `(k, d)`, like scikit-learn), and return `(scores, components)` with `scores = Xc @ components.T`, shape `(n, k)`.

**Example**: what goes in, and what should come out:
```python
Z, C = pca_svd(XP, 1)
Z.ravel()     # → [-2.121 -2.121  0.  2.121  2.121]   (or all signs flipped)
C             # → [[0.707 0.707]]   shape (1, 2)
```

In [ ]:
def pca_svd(X, k):
    pass  # your code

In [ ]:
check(8)

In [ ]:
for n in range(1, 9):
    check(n)

---
Solutions: `solutions/02_solutions.py`. Next: `03_kmeans.ipynb`.